# Verifying the narrow/broad split with ACGs

Two things from spike timing (autocorrelograms): (A) **verify** the waveform antimode narrow/broad
split against the ACGs, and (B) **cluster** cells directly from the ACG into putative IN/PY.

Verification views (Sections 4-6), keyed on the waveform label:
1. **Interpretable scatter** — trough-to-peak (the split axis) vs an ACG axis (τrise). Real
   axes, so you can read *why* the groups separate; the antimode line is drawn in.
2. **Mean ACG per group** — rate-normalized (chance = 1) so it's firing-rate-invariant, with
   95% CI bands. Narrow (IN) → sharp refractory then flat; broad (PY) → burst peak > 1.
3. **Statistics** — τrise (and the other ACG features) per group with a Mann–Whitney test.

ACG clustering (Section 7): **spectral clustering** on [firing rate, normalized mean ACG amplitude,
τrise] → two clusters → IN/PY by biophysical signature, with Mann–Whitney U (reference Fig. 5d-f),
then checked against the antimode split.

Runs on **000673**, one region pooled across sessions (`SELECT_AREAS`).

## Section 0 — Setup, config

In [ ]:
# [0.1] imports, seeds, config
import sys, os, random, warnings, time, json
from pathlib import Path
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

PROJECT = Path.cwd()                        # run this notebook from E:\SBCAT\celltyping
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from celltyping.features import build_unit_features
from celltyping.classify import assign_narrow_broad          # waveform GMM, computed locally
from celltyping.acg import acg_feature_table, spectral_cluster, NARROW, WIDE
from celltyping import viz

RS = 42
os.environ['PYTHONHASHSEED'] = str(RS)
random.seed(RS); np.random.seed(RS)
plt.rcParams['figure.dpi'] = 100

# ---- CONFIG ----
DATASET   = '000673'                        # has both waveforms and spikes
DATA_ROOT = PROJECT.parent / 'data' / DATASET
FS_HZ     = 100_000.0
SELECT_AREAS = ['hippocampus']              # region pool for THIS run (pool L+R of a region)
NB = {'narrow': '#d62728', 'broad': '#1f77b4'}   # narrow=putative IN (red), broad=putative PY (blue)
print(f'{DATASET} | region pool = {SELECT_AREAS} | seed = {RS}')

In [ ]:
# [0.2] all session files (units pooled across sessions within the region)
files = sorted(DATA_ROOT.glob('sub-*/*.nwb'))
assert files, f'no NWB files under {DATA_ROOT}'
print(f'{len(files)} session files under {DATA_ROOT}')

## Section 1 — Load + waveform GMM narrow/broad (the label under test)

In [ ]:
# [1.1] load full-session spike times + waveform/physiology features for the region pool
t0 = time.time()
df_id, spike_times = build_unit_features(files, fs_hz=FS_HZ, areas=SELECT_AREAS,
                                         return_spike_times=True, verbose=True)
df_id = df_id.reset_index(drop=True)
assert len(df_id) > 0, f'no units in {SELECT_AREAS}'
print(f'\n{len(df_id)} units from {SELECT_AREAS} in {time.time()-t0:.0f}s '
      f'| subjects={df_id.subject.nunique()} sessions={df_id.session_id.nunique()}')

In [ ]:
# [1.2] waveform GMM narrow/broad split (antimode of trough-to-peak) -- the label we verify
grp, SPLIT_MS = assign_narrow_broad(df_id, method='antimode')
df_id['wf_group'] = grp
print(f'antimode split at {SPLIT_MS:.3f} ms  (narrow=putative IN, broad=putative PY)')
print(df_id.wf_group.value_counts().to_string())
fig, ax = plt.subplots(figsize=(6.2, 3.8))
viz.plot_waveform_split(df_id, split_ms=SPLIT_MS, ax=ax)
plt.show()

## Section 2 — Build ACGs + fit τrise/τdecay (protocol steps 1-4)

Narrow (±50 ms @ 0.5 ms) → triple-exponential fit → τrise/τdecay; wide (±500 ms @ 1 ms) →
theta index. The wide ACG is the slow part (a few minutes on large pools).

In [ ]:
# [2.1] compute both ACGs per unit (returns spike-density matrices), fit, join the waveform label
t0 = time.time()
feat, Dn, Dw, lagN, lagW = acg_feature_table(df_id.unit_id.tolist(), spike_times,
                                             narrow=NARROW, wide=WIDE, verbose=True)
feat = feat.merge(df_id[['unit_id', 'session_id', 'subject', 'location', 'mean_rate_hz',
                         'burst_index', 'trough_to_peak_ms', 'half_width_ms', 'wf_valid',
                         'wf_group']], on='unit_id', how='left')
print(f'\nbuilt {len(feat)} ACGs in {time.time()-t0:.0f}s | narrow {Dn.shape}, wide {Dw.shape}')
print(f"fit R^2: median={feat.acg_r2.median():.2f}, >=0.3 in {(feat.acg_r2 >= 0.3).sum()}/{len(feat)}")
feat[['unit_id', 'wf_group', 'n_spikes', 'tau_rise', 'tau_decay', 'acg_r2', 'mean_rate_hz']].head()

In [ ]:
# [2.2] inspect example narrow-ACG fits (spike-density + fitted triple-exponential)
from celltyping.acg import compute_acg, fit_acg
ex = feat.dropna(subset=['acg_r2']).sort_values('acg_r2', ascending=False)
ex = pd.concat([ex.head(3), ex.tail(3)])
pos = lagN > 0
fig, axes = plt.subplots(2, 3, figsize=(12, 6), squeeze=False)
for ax, (_, r) in zip(axes.ravel(), ex.iterrows()):
    a = compute_acg(spike_times[df_id.index[df_id.unit_id == r.unit_id][0]], **NARROW)
    fit = fit_acg(a['density'], a['centers'])
    ax.bar(lagN[pos], a['density'][pos], width=NARROW['bin_ms'], color='#bbb')
    if fit['fit'] is not None:
        ax.plot(fit['fit_t'], fit['fit'], color=NB.get(r.wf_group, '#d62728'), lw=2)
    ax.set_title(f"{r.wf_group}  R²={r.acg_r2:.2f}  τr={r.tau_rise:.1f} ms", fontsize=8)
    ax.set_xlabel('lag (ms)')
fig.suptitle('narrow-ACG fits (top = best R², bottom = worst)', fontsize=10)
fig.tight_layout(); plt.show()

## Section 3 — QC gate

Keep units with a stable ACG fit (**R² ≥ 0.3**) *and* a narrow/broad label — the units where
both the waveform and the timing evidence exist.

In [ ]:
# [3.1] keep units with a good ACG fit AND a narrow/broad waveform label
R2_MIN     = 0.3
MIN_SPIKES = 0        # raise (e.g. 500) for stricter ACGs; reference uses only the R^2 gate
mask = ((feat.acg_r2 >= R2_MIN) & (feat.n_spikes >= MIN_SPIKES)
        & feat.tau_rise.notna() & feat.mean_rate_hz.notna()
        & feat.wf_group.isin(['narrow', 'broad'])).to_numpy()
feat_c = feat.loc[mask].reset_index(drop=True)
Dn_c, Dw_c = Dn[mask], Dw[mask]
print(f'kept {mask.sum()}/{len(feat)} units (good ACG fit + narrow/broad label)')
print(feat_c.wf_group.value_counts().to_string())

## Section 4 — Verify with interpretable axes

x = **trough-to-peak** (the waveform axis the antimode split is defined on; split drawn as a
dashed line). y = an **ACG** axis (independent evidence). If narrow and broad separate *along y
as well*, the 1-D waveform cut is corroborated by timing.

In [ ]:
# [4.1] the headline check: trough-to-peak (waveform) vs tau_rise (ACG), colored by antimode label
fig, ax = plt.subplots(figsize=(6.4, 5))
for g, c in NB.items():
    m = feat_c.wf_group == g
    ax.scatter(feat_c.loc[m, 'trough_to_peak_ms'], feat_c.loc[m, 'tau_rise'],
               s=20, alpha=0.7, color=c, label=f'{g} (n={int(m.sum())})')
ax.axvline(SPLIT_MS, color='k', ls='--', lw=1.5, label=f'antimode = {SPLIT_MS:.2f} ms')
ax.set_yscale('log')
ax.set_xlabel('trough-to-peak (ms)   [waveform axis — defines the split]')
ax.set_ylabel('ACG τrise (ms)   [independent timing axis]')
ax.set_title('Does the antimode split also separate along an ACG axis?')
ax.legend(fontsize=8); fig.tight_layout(); plt.show()

In [ ]:
# [4.2] same idea across several ACG features (trough-to-peak vs each), colored by antimode label
ACG_FEATS = ['tau_rise', 'tau_decay', 'burst_index', 'mean_rate_hz', 'mean_acg_narrow', 'theta_index']
LOGY = {'tau_rise', 'tau_decay', 'mean_rate_hz'}
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, f in zip(axes.ravel(), ACG_FEATS):
    for g, c in NB.items():
        m = feat_c.wf_group == g
        ax.scatter(feat_c.loc[m, 'trough_to_peak_ms'], feat_c.loc[m, f], s=12, alpha=0.55, color=c)
    ax.axvline(SPLIT_MS, color='k', ls='--', lw=1)
    if f in LOGY: ax.set_yscale('log')
    ax.set_xlabel('trough-to-peak (ms)'); ax.set_ylabel(f)
fig.suptitle('waveform split axis vs each ACG feature (red=narrow, blue=broad)', fontsize=11)
fig.tight_layout(); plt.show()

## Section 5 — Mean ACG per group (rate-normalized)

Average each group's ACG **divided by the unit's firing rate**, so **y = 1 is chance**
regardless of rate. Expectation: narrow (IN) shows a sharp refractory dip then a flat ~1
(tonic); broad (PY) shows a burst peak **> 1** near 3-5 ms decaying back to 1. Bands = 95% CI.

In [ ]:
# [5.1] mean rate-normalized ACG per antimode group, narrow (±50 ms) and wide (±500 ms)
rate = feat_c.mean_rate_hz.clip(lower=1e-6).to_numpy()
Rn = Dn_c / rate[:, None]        # rate-normalized narrow ACG (chance = 1)
Rw = Dw_c / rate[:, None]        # rate-normalized wide ACG
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for (lag, R, ax, ttl) in [(lagN, Rn, axes[0], 'mean narrow ACG (±50 ms)'),
                          (lagW, Rw, axes[1], 'mean wide ACG (±500 ms)')]:
    for g, c in NB.items():
        m = (feat_c.wf_group == g).to_numpy()
        mu = np.nanmean(R[m], 0)
        sem = np.nanstd(R[m], 0) / np.sqrt(max(m.sum(), 1))
        ax.plot(lag, mu, color=c, lw=2, label=f'{g} (n={int(m.sum())})')
        ax.fill_between(lag, mu - 1.96 * sem, mu + 1.96 * sem, color=c, alpha=0.2)
    ax.axhline(1.0, color='k', ls=':', lw=1)
    ax.set_title(ttl); ax.set_xlabel('lag (ms)'); ax.set_ylabel('rate-normalized (chance=1)')
    ax.legend(fontsize=8)
fig.tight_layout(); plt.show()

## Section 6 — Statistics + verdict

In [ ]:
# [6.1] per-feature narrow-vs-broad comparison (Mann-Whitney) + tau_rise violin
print(f'{"feature":16s} {"narrow med":>11s} {"broad med":>11s} {"p (MWU)":>10s}')
stats = {}
for f in ACG_FEATS:
    a = feat_c.loc[feat_c.wf_group == 'narrow', f].dropna()
    b = feat_c.loc[feat_c.wf_group == 'broad', f].dropna()
    p = mannwhitneyu(a, b).pvalue if len(a) and len(b) else np.nan
    stats[f] = dict(narrow=float(a.median()), broad=float(b.median()), p=float(p))
    print(f'{f:16s} {a.median():11.4g} {b.median():11.4g} {p:10.1e}')

data = [feat_c.loc[feat_c.wf_group == g, 'tau_rise'].dropna() for g in NB]
fig, ax = plt.subplots(figsize=(4.6, 4))
parts = ax.violinplot(data, showmedians=True)
for pc, c in zip(parts['bodies'], NB.values()):
    pc.set_facecolor(c); pc.set_alpha(0.55)
ax.set_xticks([1, 2]); ax.set_xticklabels(list(NB)); ax.set_yscale('log')
ax.set_ylabel('ACG τrise (ms)')
ax.set_title(f"τrise by antimode group  (p={stats['tau_rise']['p']:.1e})")
fig.tight_layout(); plt.show()

In [ ]:
# [6.2] verdict: do narrow (IN) units have the faster ACG rise expected of interneurons?
tr = stats['tau_rise']
direction = 'narrow < broad (expected: IN rise faster)' if tr['narrow'] < tr['broad'] \
            else 'narrow > broad (UNEXPECTED direction)'
sig = tr['p'] < 0.05
print(f"τrise: narrow={tr['narrow']:.2f} ms, broad={tr['broad']:.2f} ms  |  {direction}")
print(f"significant (p<0.05): {sig}  (p={tr['p']:.1e})")
n_sig = sum(v['p'] < 0.05 for v in stats.values() if np.isfinite(v['p']))
print(f"\nACG features differing between groups (p<0.05): {n_sig}/{len(stats)}")
print('VERDICT:', 'antimode split is corroborated by spike timing.'
      if (sig and tr['narrow'] < tr['broad']) else
      'timing does NOT clearly corroborate the split — inspect the panels above.')

## Section 7 — ACG spectral clustering → putative IN / PY

Cluster the cells directly from spike timing (independent of the waveform), following the
reference: **spectral clustering** on **[firing rate, normalized mean ACG amplitude, τrise]** into
two groups, then assign identities by biophysical signature — the cluster with **higher firing
rate, greater normalized mean ACG amplitude, and larger τrise** is **putative interneurons (IN)**;
the other is **putative pyramidal (PY)**. Mann–Whitney U on the three features quantifies the split.

In [ ]:
# [7.1] spectral clustering on the reference's 3 features -> two clusters -> IN / PY
CLUST_FEATS = ['log10_rate', 'mean_acg_narrow', 'tau_rise']   # firing rate, norm mean ACG amp, τrise
feat_c['log10_rate'] = np.log10(feat_c.mean_rate_hz.clip(lower=1e-6))
Fc = feat_c[CLUST_FEATS].to_numpy(float)
med = np.nanmedian(Fc, axis=0); nanpos = np.where(np.isnan(Fc)); Fc[nanpos] = np.take(med, nanpos[1])
labels, _ = spectral_cluster(Fc, n_clusters=2, n_neighbors=15, seed=RS)   # z-scores internally
feat_c['acg_cluster'] = labels

# biophysical assignment: IN = higher rate + higher mean-ACG amplitude + larger τrise
def _zc(x):
    x = np.asarray(x, float); s = np.nanstd(x)
    return (x - np.nanmean(x)) / s if s > 0 else np.zeros_like(x)
inness = _zc(feat_c.log10_rate) + _zc(feat_c.mean_acg_narrow) + _zc(feat_c.tau_rise)
cl_mean = {c: float(np.nanmean(inness[labels == c])) for c in np.unique(labels)}
in_c = max(cl_mean, key=cl_mean.get)
feat_c['acg_type'] = np.where(labels == in_c, 'IN', 'PY')
ACG_COL = {'IN': '#d62728', 'PY': '#1f77b4'}
print(feat_c.acg_type.value_counts().to_string())

In [ ]:
# [7.2] biophysical signatures of the two clusters (Mann-Whitney U; reference Fig. 5d-f)
SIG_FEATS = [('mean_rate_hz', 'firing rate (Hz)', True),
             ('mean_acg_narrow', 'norm. mean ACG amplitude', False),
             ('tau_rise', 'τrise (ms)', True)]
IN = feat_c[feat_c.acg_type == 'IN']; PY = feat_c[feat_c.acg_type == 'PY']
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (f, lab, logy) in zip(axes, SIG_FEATS):
    a = IN[f].dropna(); b = PY[f].dropna()
    U, p = mannwhitneyu(a, b)
    parts = ax.violinplot([a, b], showmedians=True)
    for pc, c in zip(parts['bodies'], [ACG_COL['IN'], ACG_COL['PY']]):
        pc.set_facecolor(c); pc.set_alpha(0.6)
    if logy: ax.set_yscale('log')
    ax.set_xticks([1, 2]); ax.set_xticklabels([f'IN\n(n={len(a)})', f'PY\n(n={len(b)})'])
    ax.set_ylabel(lab); ax.set_title(f'U={U:.0f}, p={p:.2g}', fontsize=9)
    print(f'{lab:26s} IN med={a.median():.4g}  PY med={b.median():.4g}  U={U:.0f}  p={p:.3g}')
fig.suptitle(f'ACG clusters — IN (n={len(IN)}) vs PY (n={len(PY)}); '
             'IN = higher rate, higher mean ACG, larger τrise', fontsize=10)
fig.tight_layout(); plt.show()

In [ ]:
# [7.3] does the ACG clustering agree with the waveform antimode split?
from sklearn.metrics import adjusted_rand_score
ct = pd.crosstab(feat_c.acg_type, feat_c.wf_group)
print('ACG cluster × waveform antimode:'); print(ct.to_string())
agree = ((feat_c.acg_type == 'IN') == (feat_c.wf_group == 'narrow')).mean()
ari = adjusted_rand_score(feat_c.wf_group, feat_c.acg_type)
print(f'\nagreement (IN<->narrow, PY<->broad): {agree:.0%}  |  ARI = {ari:.2f}')

## Section 8 — Save per-unit table + provenance

In [ ]:
# [8.1] save the waveform label + ACG cluster + ACG features per unit
region_tag = '+'.join(a.lower() for a in SELECT_AREAS)
OUT_COLS = ['unit_id', 'subject', 'session_id', 'location', 'wf_group', 'acg_cluster', 'acg_type',
            'trough_to_peak_ms', 'half_width_ms', 'tau_rise', 'tau_decay', 'acg_r2', 'refrac_ms',
            'mean_acg_narrow', 'mean_acg_wide', 'theta_index', 'mean_rate_hz', 'burst_index', 'n_spikes']
out_df = feat_c[[c for c in OUT_COLS if c in feat_c.columns]].copy()
out = PROJECT / 'outputs' / 'celltype' / f'unit_labels_acg_{region_tag}.csv'
out.parent.mkdir(parents=True, exist_ok=True)
out_df.to_csv(out, index=False)
prov = dict(dataset=DATASET, areas=SELECT_AREAS, n_units=int(len(out_df)),
            wf_split_ms=float(SPLIT_MS), r2_min=R2_MIN, seed=RS,
            cluster_feats=CLUST_FEATS,
            acg_type_counts={k: int(v) for k, v in out_df.acg_type.value_counts().items()},
            group_counts={k: int(v) for k, v in out_df.wf_group.value_counts().items()},
            feature_stats=stats)
out.with_suffix('.json').write_text(json.dumps(prov, indent=2))
print(f'wrote {out}  ({len(out_df)} units) + provenance {out.with_suffix(".json").name}')
out_df.head(12)

## Reading the result

- **`[4.1]`** is the core check: if the red (narrow) and blue (broad) points land at different
  heights (τrise) as well as on opposite sides of the antimode line, the 1-D waveform split has
  an independent timing correlate. `[4.2]` shows the same against other ACG features.
- **`[5.1]`** should show narrow (IN) with a flat ~1 tail (tonic) and broad (PY) with a burst
  peak > 1 — the classic cell-type ACG contrast, firing-rate-invariant.
- **`[6.x]`** turns it into numbers: τrise narrow-vs-broad and how many ACG features differ.
- **`[7.x]`** is the ACG-only clustering: two clusters labeled IN/PY by biophysical signature
  (higher rate + higher mean ACG amplitude + larger τrise = IN), Mann–Whitney U per feature, and
  its agreement (ARI) with the waveform antimode split.

If timing and waveform disagree, that's a real finding (the width split may be catching
something other than the IN/PY axis in this region), not a bug. Output
`outputs/celltype/unit_labels_acg_<region>.csv` carries the waveform label, the ACG cluster
(`acg_type` IN/PY), and the ACG features per `unit_id`; the `.json` sidecar stores the stats.